# Chapter 18: Explainability and Trustworthy AI
**Part V — AI Ethics, Governance & Law**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

LIME, SHAP, ISO/IEC 42001, and the trade-off between accuracy and interpretability.

## Learning Objectives
See Chapter 18 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Train a model


In [ ]:
import shap
import xgboost as xgb

# Train a model
model = xgb.XGBClassifier().fit(X_train, y_train)

# Compute SHAP values
explainer = shap.Explainer(model)
shap_values = explainer(X_test)

# Summary plot: feature importance across all predictions
shap.summary_plot(shap_values, X_test)

# Waterfall plot: explanation for a single prediction
shap.waterfall_plot(shap_values[0])

# Force plot: visualise feature contributions
shap.force_plot(explainer.expected_value,
                shap_values.values[0], X_test.iloc[0])

## EU AI Act risk classification checklist (pseudocode)


In [ ]:
# EU AI Act risk classification checklist (pseudocode)

def classify_ai_system(use_case, deployment_context):
    prohibited = [
        "social scoring", "subliminal manipulation",
        "real-time biometric surveillance"
    ]
    high_risk_sectors = [
        "critical infrastructure", "employment", "education",
        "credit scoring", "law enforcement", "migration",
        "administration of justice", "health and safety"
    ]
    if any(p in use_case for p in prohibited):
        return "PROHIBITED - do not deploy"
    elif any(s in deployment_context for s in high_risk_sectors):
        return "HIGH RISK - full compliance required"
    elif "interacts with humans" in use_case:
        return "LIMITED RISK - transparency obligations"
    else:
        return "MINIMAL RISK - no specific obligations"

## SHAP values on a Random Forest — global and local explanations


In [ ]:
# SHAP values on a Random Forest — global and local explanations
# Install: pip install shap xgboost
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("shap not installed. Run: pip install shap")

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f"Test accuracy: {clf.score(X_test, y_test):.4f}")

if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test)
    
    # Global feature importance
    import matplotlib.pyplot as plt
    shap_abs = np.abs(shap_values[1]).mean(axis=0)
    feat_imp = pd.Series(shap_abs, index=data.feature_names).sort_values(ascending=True)
    
    plt.figure(figsize=(10, 7))
    feat_imp.tail(15).plot(kind='barh', color='#2E75B6', edgecolor='white')
    plt.xlabel('Mean |SHAP value| (average impact on model output)')
    plt.title('Global Feature Importance via SHAP\n(Breast Cancer Classification)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ch18_shap_global.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Local explanation for one prediction
    idx = 0
    pred_class = clf.predict(X_test.iloc[[idx]])[0]
    pred_prob  = clf.predict_proba(X_test.iloc[[idx]])[0, 1]
    print(f"\nSample #{idx}: prediction = {'malignant' if pred_class==1 else 'benign'} (prob={pred_prob:.3f})")
    
    sv = shap_values[1][idx]
    contrib = pd.Series(sv, index=data.feature_names).sort_values(key=abs, ascending=False).head(8)
    print("\nTop SHAP contributions for this prediction:")
    for feat, val in contrib.items():
        direction = "↑ towards malignant" if val > 0 else "↓ towards benign"
        print(f"  {feat:<35} {val:>+.4f}  {direction}")
else:
    print("Install shap to run this example: pip install shap")

---

## 📚 Review Questions

See Chapter 18 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 18 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*